# Сравнение моделей: три варианта `ratio(s, d)`

Этот ноутбук показывает, как меняется поведение байесовской модели оценок при разных формулах «совпадения» между способностью студента `s ∈ [0, 1]` и сложностью предмета `d ∈ [0, 1]`. Сами `f_2..f_5` и приоры (chi-square per-item) одинаковые во всех трёх вариантах — отличается только `ratio`.

## Три формулы

| вариант | формула | свойства |
| --- | --- | --- |
| `legacy_sd` | `s / (s + d)` | симметричен относительно `s ↔ d`; точка ½ — когда `s = d` |
| `current`   | `1 − d (1 − s)` | стремится к 1, если `d = 0` ИЛИ `s = 1`; асимметричен |
| `sigmoid`   | `σ(k(s − d))`, `k=5` | гладкая S-кривая; параметр `k` управляет резкостью |

Семантика, которой надо матчиться: «бездарь+сложный → 2», «средний+средний → 3-4», «умный+лёгкий → 5».
Из таблицы ниже видно, что `legacy_sd` и `sigmoid` дают это «из коробки», а `current` тащит среднюю точку к 4-5.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from utils import ratio, f_2, f_3, f_4, f_5, RATIO_KINDS
import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

## Угловые точки

Считаем ожидаемую оценку `E[grade | s, d]` в пяти ключевых точках для каждого варианта.

In [ ]:
def expected_grade(s, d, kind):
    r = ratio(s, d, kind=kind)
    return 2*f_2(r) + 3*f_3(r) + 4*f_4(r) + 5*f_5(r)

corners = {
    '(s=0, d=1)  бездарь + сложный': (0.0, 1.0),
    '(s=0.5, d=0.5) средний + средний': (0.5, 0.5),
    '(s=1, d=0)  умный + лёгкий': (1.0, 0.0),
    '(s=0, d=0)  бездарь + лёгкий': (0.0, 0.0),
    '(s=1, d=1)  умный + сложный': (1.0, 1.0),
}
rows = []
for label, (s, d) in corners.items():
    row = {'сценарий': label}
    for kind in RATIO_KINDS:
        row[kind] = round(float(expected_grade(s, d, kind)), 2)
    rows.append(row)
pd.DataFrame(rows).set_index('сценарий')

## Поверхности `ratio(s, d)` и `E[grade | s, d]`

Сверху — само значение `ratio`, снизу — соответствующая ожидаемая оценка (от 2 до 5). Белые точки — пять угловых сценариев из таблицы выше.

Видно, что у **current** правый-верхний и левый-нижний углы оба «жёлто-зелёные» (≈5), потому что `d(1−s) = 0` в обоих случаях. У **legacy_sd** и **sigmoid** середина «по-настоящему» средняя (≈3.5).

In [ ]:
display(Image('../results/plots/ratio_surfaces.png'))

## Профиль E[grade] как функция способности студента

Три среза: вдоль диагонали `s = d`, при лёгком предмете `d = 0.3`, при сложном `d = 0.8`. Серые пунктиры — границы оценок 2 и 5.

In [ ]:
display(Image('../results/plots/ratio_curves.png'))

## Функции `f_i` (legacy vs current)

Для контекста — как менялись сами вероятности оценок при правке степеней (старые 76, 12 → новые 3, 2).

In [ ]:
display(Image('../results/plots/f_functions_legacy_vs_current.png'))

## Постериоры `item_difficulty` по предметам

MCMC: chains=4, tune=2000, draws=4000, target_accept=0.95, chi-square per-item приор.
Бар-чарты — апостериорное среднее по каждому предмету для всех трёх ratio, по семестрам/моделям.

In [ ]:
display(Image('../results/plots/ratio_posterior_compare.png'))

## HDI 94% для semester 1

Три панели — три варианта ratio; одинаковая ось Y чтобы сравнение было поточечным.

In [ ]:
display(Markdown('### Модель `cond` (с гейтом на 2 и 5)'))
display(Image('../results/plots/items_hdi_sem1_cond.png'))

In [ ]:
display(Markdown('### Модель `nocond` (только f-функции)'))
display(Image('../results/plots/items_hdi_sem1_nocond.png'))

## Краткие выводы

1. **Среднее по средне-средним предметам.** `current` (1−d(1−s)) на (0.5, 0.5) даёт E[grade] ≈ 4.24 — то есть «обычный студент на обычном предмете» ожидаемо получает четыре с плюсом. У `legacy_sd` и `sigmoid` E[grade] (0.5, 0.5) ≈ 3.5, что соответствует словесному ожиданию «3-4».

2. **Углы (0,0) и (1,1).** У `current` оба угла «слипаются» — `d(1−s) = 0` в обоих, поэтому `ratio = 1`, и модель уверенно ставит 5. У двух симметричных вариантов углы тоже логичны: 3-4 (бездарь на лёгком — неопределённо; умный на сложном — тоже неопределённо).

3. **Гейт в `cond` был сломан.** Веса считались как `p_5 = σ_5 / (σ_5 + σ_2)` и `p_2 = σ_2 / (σ_5 + σ_2)`, то есть нормировались друг на друга. Из-за этого `p_5 + p_2 ≡ 1`, вес `p_middle` тождественно зануляется, и семейство `f_2..f_5` не участвовало в правдоподобии вообще: `cond` вырождалась в выбор между 2 и 5 и физически не могла предсказать 3 или 4. На середине шкалы она выдавала `[0.499, 0.001, 0.001, 0.499]` — подбрасывание монетки между двойкой и пятёркой. Нормировку убрали, веса берутся из сигмоид напрямую. Регрессия закрыта тестом `test_gate_is_a_real_mixture`.

4. **Пороги гейта пришлось сдвинуть внутрь.** После фикса выяснилось, что с «естественными» порогами `0.1 / 0.9` гейт срабатывает только там, где `f`-функции и без него дают 0.997 — то есть `cond` становится численно неотличима от `nocond` (расхождение в третьем знаке). Дефолт изменён на `gate_lo=0.3`, `gate_hi=0.7`, `gate_temp=15`: теперь гейт влияет на промежуточные случаи (хороший студент на среднем предмете идёт с E=4.17 на 4.67), но середина шкалы сохраняется — при `s = d` тройка и четвёрка остаются самыми вероятными.

5. **Выбор по умолчанию: `sigmoid` + `cond`.** `sigmoid` симметричен и даёт правильную семантику углов, `k` регулирует резкость. Оговорка: по сходимости `sigmoid` — самый тяжёлый из трёх вариантов, поэтому число draws поднято (см. раздел о сходимости в README). Остальные комбинации не удалены — MCMC считает все 3 × 2, любую можно запросить через `scripts/query.py --ratio ... --model ...`.